# Análise de Embeddings

Este notebook parte de um corpus de textos curtos, gera embeddings com um modelo pré-treinado e investiga a estrutura do espaço resultante. A pergunta que organiza o trabalho é metodológica: quando os dados chegam já em forma vetorial e de dimensão alta, como decidir quantos grupos existem, em que espaço agrupá-los, e como saber se o resultado significa alguma coisa. O corpus foi montado com quatro temas conhecidos, o que permite confrontar cada decisão com a resposta certa.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, confusion_matrix, silhouette_score
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## O Corpus

Oitenta e oito frases curtas em inglês, distribuídas igualmente entre quatro temas: culinária, geografia, inteligência artificial e finanças. Os rótulos existem apenas para avaliar os resultados no fim, e nenhum passo da análise os utiliza.

In [ ]:
sentences = [
    # culinária
    'I swap butter for olive oil in many recipes.',
    'I prefer my coffee with no sugar and a splash of milk.',
    'The recipe for pasta carbonara is simple.',
    'A pinch of salt enhances sweetness in desserts.',
    'Aromatics like garlic and onion build flavor early.',
    'Fermented foods add acidity and complexity.',
    'Marinating tofu improves texture and taste.',
    'Deglazing lifts browned bits to make pan sauces.',
    'Tempering chocolate stabilizes cocoa butter crystals.',
    'I batch-cook grains for quick lunches.',
    'Resting steak helps redistribute the juices.',
    'Sourdough starter needs regular feedings to stay active.',
    'Umami-rich ingredients deepen savory dishes.',
    'Al dente pasta retains a slight bite after cooking.',
    'Stir-frying requires high heat and constant movement.',
    'I cook vegetarian meals on weekdays to simplify planning.',
    'Sous-vide delivers precise temperature control.',
    'Mise en place speeds up weeknight cooking.',
    'I keep a jar of homemade pesto for pasta.',
    'I like to cook Italian dishes on Sundays.',
    'Roasting vegetables caramelizes natural sugars.',
    'Proofing time affects a bread’s crumb structure.',
    # geografia
    'Canberra is the capital of Australia.',
    'Ottawa is the capital city of Canada.',
    'Paris is the most populated city in France.',
    'Tokyo is among the most populous metropolitan areas worldwide.',
    'The Sahara Desert spans much of North Africa.',
    'The Great Barrier Reef lies off Australia’s northeast coast.',
    'Iceland lies on the Mid-Atlantic Ridge.',
    'The Baltic states border the eastern Baltic Sea.',
    'What is the capital of France?',
    'Johannesburg is a major city but not South Africa’s capital.',
    'The Danube passes through multiple European capitals.',
    'The Amazon River carries one of the largest water volumes on Earth.',
    'The Atacama is one of the driest deserts on the planet.',
    'Mount Everest is the highest peak above sea level.',
    'What country contains the city of Kyoto?',
    'The Nile flows northward into the Mediterranean Sea.',
    'The Alps stretch across several central European countries.',
    'The Andes form a continuous mountain range along South America.',
    'The Caspian Sea is a landlocked body of water.',
    'Cairo sits along the Nile River delta.',
    'Lagos is Nigeria’s largest city by population.',
    'New Delhi serves as the seat of India’s government.',
    # inteligência artificial
    'Alignment techniques reduce harmful outputs.',
    'Explainable AI highlights salient features for decisions.',
    'Transformer models enable long-range language dependencies.',
    'Quantization reduces memory with minimal accuracy loss.',
    'Vector databases power semantic search at scale.',
    'Distillation transfers knowledge from large to small models.',
    'Retrieval-augmented generation grounds answers in sources.',
    'Multimodal learning aligns text with images and audio.',
    'Reinforcement learning fine-tunes policies from human feedback.',
    'Edge AI runs models under strict latency constraints.',
    'Graph neural networks capture relational structure.',
    'Continual learning mitigates catastrophic forgetting.',
    'Diffusion models synthesize high-fidelity images.',
    'Self-supervised pretraining reduces labeled data needs.',
    'Causal inference distinguishes correlation from effect.',
    'Prompt engineering steers generative behavior reliably.',
    'Few-shot prompting improves generalization on new tasks.',
    'Natural language processing has advanced greatly.',
    'Artificial intelligence is transforming the world.',
    'Evaluation with benchmarks must avoid data leakage.',
    'Federated learning trains models without centralizing data.',
    'LoRA adapters enable efficient fine-tuning.',
    # finanças
    'Black swan events stress-test portfolio resilience.',
    'Inflation erodes real purchasing power of cash.',
    'Value stocks trade at lower multiples relative to fundamentals.',
    'Tax-loss harvesting offsets capital gains.',
    'Investing in technology can be risky.',
    'Risk tolerance should guide position sizing.',
    'Time in the market beats timing the market.',
    'Behavioral biases can derail investment plans.',
    'A healthy emergency fund reduces forced selling.',
    'Liquidity risk rises when trading volumes are thin.',
    'The stock market experienced a drop today.',
    'Rebalancing restores target asset allocation.',
    'Bond duration measures sensitivity to interest-rate changes.',
    'Expense ratios compound against long-term returns.',
    'Covered calls generate income with capped upside.',
    'Growth investing prioritizes earnings expansion.',
    'Diversification reduces idiosyncratic risk across holdings.',
    'Sharpe ratio evaluates risk-adjusted performance.',
    'Credit spreads widen during economic uncertainty.',
    'Emerging markets add diversification but higher volatility.',
    'Dollar-cost averaging smooths entry price over time.',
    'ETFs provide broad market exposure with intraday liquidity.',
]


themes = ['culinária', 'geografia', 'inteligência artificial', 'finanças']
y_true = np.repeat(np.arange(4), 22)

print(f"{len(sentences)} frases, {len(themes)} temas, {np.bincount(y_true)} por tema")

## Gerando os Embeddings

O `all-MiniLM-L6-v2` produz vetores de 384 dimensões já normalizados para norma 1, o que faz do produto interno a própria similaridade de cosseno.

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
X = model.encode(sentences)

print(f"X: {X.shape}")
print(f"normas: min={np.linalg.norm(X, axis=1).min():.4f}, max={np.linalg.norm(X, axis=1).max():.4f}")

## A Geometria em 384 Dimensões

Antes de projetar ou agrupar, vale olhar para a matriz de similaridade completa. Se os temas formam grupos coesos, a similaridade média dentro de um tema deve ser maior que a similaridade entre temas diferentes.

In [ ]:
similarity = X @ X.T

block_means = np.zeros((4, 4))
for a in range(4):
    for b in range(4):
        block = similarity[np.ix_(y_true == a, y_true == b)]
        # dentro do tema, exclui a diagonal (similaridade de cada frase consigo mesma)
        block_means[a, b] = block[np.triu_indices(22, k=1)].mean() if a == b else block.mean()

print("cosseno médio por bloco:")
print(f"{'':>24}" + "".join(f"{t[:10]:>12}" for t in themes))
for a, theme in enumerate(themes):
    print(f"{theme:>24}" + "".join(f"{v:>12.3f}" for v in block_means[a]))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im = ax1.imshow(similarity, cmap='viridis')
for boundary in [22, 44, 66]:
    ax1.axhline(boundary - 0.5, color='white', lw=1)
    ax1.axvline(boundary - 0.5, color='white', lw=1)
ax1.set(title='Matriz de similaridade (ordenada por tema)', xticks=[], yticks=[])
fig.colorbar(im, ax=ax1, fraction=0.046)

same = similarity[np.triu_indices(88, k=1)][
    (y_true[:, None] == y_true[None, :])[np.triu_indices(88, k=1)]]
other = similarity[np.triu_indices(88, k=1)][
    (y_true[:, None] != y_true[None, :])[np.triu_indices(88, k=1)]]
ax2.hist(same, bins=30, histtype='step', lw=2, density=True, label='mesmo tema')
ax2.hist(other, bins=30, histtype='step', lw=2, density=True, label='temas diferentes')
ax2.set(xlabel='similaridade de cosseno', ylabel='densidade')
ax2.legend()
plt.tight_layout()
plt.show()

Os quatro blocos da diagonal são visivelmente mais claros que o resto. Numericamente, a similaridade média dentro do tema vai de 0,149 (geografia) a 0,266 (finanças), enquanto entre temas ela fica entre $-0{,}003$ e 0,075.

Dois detalhes valem registro. Geografia é o tema menos coeso, o que faz sentido: capitais, rios, desertos e montanhas compartilham pouco vocabulário entre si. E o par mais próximo entre temas distintos é inteligência artificial com finanças, 0,075, dois domínios que dividem termos como "modelo", "risco" e "performance".

Os histogramas se sobrepõem, ou seja, nenhum limiar único de cosseno separa perfeitamente pares do mesmo tema de pares de temas diferentes. A separação existe na distribuição, não em cada par individualmente.

## Projeções

Três projeções em duas dimensões, coloridas pelos temas verdadeiros. As cores servem só para nós avaliarmos o resultado; os algoritmos não as receberam.

In [ ]:
pca = PCA(n_components=2).fit(X)
projections = {
    f'PCA ({pca.explained_variance_ratio_.sum():.1%} da variância)': pca.transform(X),
    't-SNE (perplexity=10)': TSNE(n_components=2, perplexity=10, init='pca',
                                  random_state=42).fit_transform(X),
    'UMAP (n_neighbors=10)': umap.UMAP(n_neighbors=10, min_dist=0.1,
                                       random_state=42).fit_transform(X),
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, (title, Z) in zip(axes, projections.items()):
    for c, theme in enumerate(themes):
        ax.scatter(*Z[y_true == c].T, s=35, label=theme)
    ax.set(title=title, xticks=[], yticks=[])
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

As três projeções concordam sobre a existência de quatro grupos, mas divergem sobre a geometria. O PCA retém apenas 14,1% da variância e deixa os grupos próximos e parcialmente sobrepostos. O t-SNE os separa por espaço vazio. O UMAP os separa mais ainda e os compacta.

Nenhuma dessas três impressões visuais é evidência de que existem exatamente quatro grupos: espaço vazio entre aglomerados é uma característica do t-SNE e do UMAP, que aparece mesmo em dados sem estrutura de grupo. A decisão sobre o número de grupos precisa ser tomada com uma medida, não com o gráfico.

## Quantos Grupos?

O critério é a silhueta média em função de $k$, calculada no espaço original de 384 dimensões. Como conhecemos os rótulos, podemos acompanhar em paralelo o índice de Rand ajustado (ARI), que mede a concordância entre o agrupamento e a verdade, um recurso que não estaria disponível em uma análise real.

In [ ]:
scores = []
for k in range(2, 9):
    labels = KMeans(k, n_init=10, random_state=42).fit_predict(X)
    scores.append((k, silhouette_score(X, labels), adjusted_rand_score(y_true, labels)))

print(f"{'k':>3} {'silhueta':>10} {'ARI':>8}")
for k, sil, ari in scores:
    print(f"{k:>3} {sil:>10.4f} {ari:>8.4f}")

In [ ]:
ks, sils, aris = zip(*scores)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ks, sils, 'o-', label='silhueta (sem rótulos)')
ax.plot(ks, aris, 's--', label='ARI (com rótulos)')
ax.axvline(4, color='crimson', ls=':', label='k verdadeiro')
ax.set(xlabel='k', ylabel='valor')
ax.legend()
plt.show()

A silhueta atinge o máximo exatamente em $k=4$, com 0,0861, e o ARI nesse ponto é 1,000: o K-Means recuperou os quatro temas sem errar uma única frase.

O valor absoluto da silhueta merece comentário. Em 384 dimensões, 0,0861 é um número que, lido isoladamente, sugeriria ausência total de estrutura de grupo. Ele é baixo porque a silhueta compara distâncias, e em dimensão alta as distâncias entre pontos se concentram em uma faixa estreita: mesmo grupos perfeitamente separados produzem silhuetas próximas de zero. **O que informa é onde está o máximo, não o valor no máximo.**

## Agrupar Antes ou Depois de Projetar?

Uma prática comum é reduzir para duas dimensões e agrupar na projeção, porque os grupos ficam visualmente óbvios ali. Vale medir o que essa escolha custa.

In [ ]:
print(f"{'espaço':>26} {'silhueta':>10} {'ARI':>8}")
for name, Z in [('384 dimensões originais', X)] + list(projections.items()):
    labels = KMeans(4, n_init=10, random_state=42).fit_predict(Z)
    print(f"{name.split(' (')[0]:>26} {silhouette_score(Z, labels):>10.4f} "
          f"{adjusted_rand_score(y_true, labels):>8.4f}")

A tabela inverte a intuição. Agrupar nas projeções produz silhuetas muito maiores, com 0,660 no PCA, 0,692 no t-SNE e 0,838 no UMAP contra 0,086 no espaço original, e ao mesmo tempo agrupamentos **piores**: ARI 0,911, 0,969 e 0,969, contra 1,000 no espaço original.

As duas coisas têm a mesma causa. O t-SNE e o UMAP produzem, por construção, aglomerados compactos separados por espaço vazio; é exatamente essa a configuração que maximiza a silhueta. A medida está avaliando o resultado da projeção, não a estrutura dos dados. Enquanto isso, comprimir 384 dimensões em 2 descarta informação, e as poucas frases ambíguas acabam do lado errado.

A conclusão prática: **agrupe no espaço original e use a projeção apenas para visualizar o agrupamento**. E não compare silhuetas calculadas em espaços diferentes, porque o número só é comparável entre agrupamentos do mesmo conjunto de coordenadas.

In [ ]:
labels = KMeans(4, n_init=10, random_state=42).fit_predict(X)

Z = projections['UMAP (n_neighbors=10)']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, coloring, title in [(ax1, y_true, 'temas verdadeiros'),
                            (ax2, labels, 'clusters do K-Means em 384D')]:
    ax.scatter(Z[:, 0], Z[:, 1], c=coloring, cmap='tab10', s=40)
    ax.set(title=title, xticks=[], yticks=[])
plt.tight_layout()
plt.show()

print(confusion_matrix(y_true, labels))

## Rotulando os Clusters

O K-Means devolve números, não nomes. Uma forma direta de descrever cada grupo é listar as frases mais próximas do seu centróide, que funcionam como representantes.

In [ ]:
centroids = KMeans(4, n_init=10, random_state=42).fit(X).cluster_centers_
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)

for c, centroid in enumerate(centroids):
    print(f"--- cluster {c} ({np.sum(labels == c)} frases) ---")
    for i in np.argsort(X @ centroid)[::-1][:3]:
        print(f"  {X[i] @ centroid:.4f}  {sentences[i]}")

Os representantes identificam os quatro temas sem ambiguidade. Note que eles não são necessariamente as frases mais *típicas* no sentido intuitivo: o representante de geografia é "What country contains the city of Kyoto?", uma pergunta, porque o centróide é a média de um grupo que mistura perguntas e afirmações.

## Classificação por Centróide

Com os grupos identificados e nomeados, classificar um texto novo é calcular o seu embedding e escolher o centróide mais próximo. Para avaliar honestamente esse classificador usamos *leave-one-out*: cada frase é classificada por centróides recalculados sem ela.

In [ ]:
def theme_centroids(X, y, exclude=None):
    """Centróides normalizados de cada tema, opcionalmente excluindo uma amostra."""
    mask = np.ones(len(X), dtype=bool)
    if exclude is not None:
        mask[exclude] = False
    centroids = np.array([X[mask & (y == c)].mean(axis=0) for c in range(4)])
    return centroids / np.linalg.norm(centroids, axis=1, keepdims=True)


predictions = np.array([np.argmax(theme_centroids(X, y_true, exclude=i) @ X[i])
                        for i in range(len(X))])

print(f"acurácia leave-one-out (centróide): {(predictions == y_true).mean():.4f}")
print(f"acurácia leave-one-out (k-NN, k=5): "
      f"{cross_val_score(KNeighborsClassifier(5), X, y_true, cv=LeaveOneOut()).mean():.4f}")

for i in np.where(predictions != y_true)[0]:
    print(f"\nerro: {sentences[i]!r}")
    print(f"  verdadeiro={themes[y_true[i]]}, previsto={themes[predictions[i]]}")
    print(f"  scores={dict(zip(themes, np.round(theme_centroids(X, y_true, exclude=i) @ X[i], 3)))}")

Ambos os classificadores acertam 87 das 88 frases, 98,86%. O único erro é revelador: *"Causal inference distinguishes correlation from effect."* foi atribuída a finanças (0,241) em vez de inteligência artificial (0,226). A diferença entre os dois scores é de 0,015, e a frase é genuinamente ambígua, porque inferência causal é um tema de estatística aplicada tanto a modelos quanto a mercados. O erro está no rótulo que impusemos, não no modelo.

In [ ]:
def classify(text, centroids, themes):
    """Classifica um texto pelo centróide de tema mais próximo."""
    scores = centroids @ model.encode([text])[0]
    return themes[int(np.argmax(scores))], float(scores.max()), scores


final_centroids = theme_centroids(X, y_true)

for text in ["Index funds usually beat active managers over decades.",
             "Add the garlic only after the onions turn translucent.",
             "The Rhine flows through Germany and the Netherlands.",
             "Attention layers scale quadratically with sequence length.",
             "My cat sleeps on the keyboard all afternoon."]:
    theme, score, _ = classify(text, final_centroids, themes)
    print(f"{theme:>24} ({score:.3f})  {text}")

As quatro primeiras frases caem no tema certo com scores entre 0,307 e 0,463. A quinta, sobre um gato dormindo no teclado, não pertence a nenhum dos quatro temas, e mesmo assim recebe um rótulo, culinária, porque o classificador sempre devolve o argmax de alguma coisa.

O que a distingue é o score: 0,160, bem abaixo dos demais. Um classificador por centróide precisa de um limiar de rejeição para ser utilizável, e esse limiar se calibra medindo a distribuição de scores em textos que sabidamente pertencem aos temas, não escolhendo um número redondo.

## Custo e Limitações

A vetorização do corpus é o passo caro e acontece uma única vez: uma passagem do Transformer por frase. Tudo o que veio depois, incluindo matriz de similaridade, K-Means, projeções e classificação, opera sobre 88 vetores de 384 dimensões e é instantâneo. Em escala maior, a matriz de similaridade completa é $O(N^2)$ em memória e passa a ser o gargalo antes do modelo.

O resultado desta análise é bom demais para ser representativo. São quatro temas muito distintos, 22 frases cada, frases curtas e bem escritas, e ARI 1,000. Corpora reais têm temas sobrepostos, distribuição desbalanceada e textos ambíguos, e nada disso apareceu aqui.

A silhueta em dimensão alta também é frágil. Ela funcionou como critério para escolher $k$, mas com valor absoluto próximo de zero, e em um corpus com estrutura menos nítida o máximo pode cair no lugar errado, o que torna prudente confirmar a escolha com outro critério. Some-se a isso que o K-Means impõe grupos esféricos e de tamanho parecido, forma que este corpus tem exatamente por construção: temas com número muito desigual de textos ou com subestrutura interna não seriam recuperados tão bem.

Por fim, os rótulos foram impostos por nós, e a única frase classificada "errada" mostra que a fronteira entre inteligência artificial e finanças é uma decisão nossa, não um fato do corpus. A escala dos scores depende do modelo, de modo que os valores de similaridade e o limiar de rejeição precisam ser recalibrados ao trocar de modelo de embedding.

## Exercícios

### Exercício 1: Um Quinto Tema

Acrescente ao corpus 22 frases de um tema novo, próximo de um dos existentes, como esportes, saúde ou música. Repita a análise completa: curva de silhueta, agrupamento com o $k$ escolhido e ARI contra os cinco rótulos. A silhueta ainda aponta o $k$ correto? O tema novo é separado do tema vizinho, ou os dois se fundem?

In [ ]:
# Seu código aqui

### Exercício 2: Limiar de Rejeição

Escreva uma versão de `classify` que devolva `None` quando o maior score ficar abaixo de um limiar. Calibre esse limiar usando a distribuição de scores leave-one-out das 88 frases do corpus, e avalie a função em um conjunto de teste que você monte com metade de frases dos quatro temas e metade de frases de assuntos totalmente distintos. Reporte quantas frases de cada tipo foram aceitas e rejeitadas, e discuta o compromisso entre rejeitar textos válidos e aceitar textos fora do domínio.

In [ ]:
# Seu código aqui

### Exercício 3: Agrupamento Hierárquico

Substitua o K-Means por `sklearn.cluster.AgglomerativeClustering` com `metric='cosine'` e compare os quatro tipos de ligação (`'single'`, `'complete'`, `'average'` e `'ward'`, sendo que este último só aceita distância euclidiana). Para cada um, meça o ARI com quatro grupos e plote o dendrograma. Em seguida, corte a árvore em oito grupos e verifique se as subdivisões dentro de cada tema fazem sentido semântico.

In [ ]:
# Seu código aqui